## gold_analysis
Objective: Create a flat, feature-rich table by joining datasets and engineering new attributes for Machine Learning.

#### Setup read/write I/O

In [0]:
from pyspark.sql import functions as F

SILVER = "/Volumes/workspace/cms_hospital/silver"
GOLD = "/Volumes/workspace/cms_hospital/gold"

def read_delta(path):
    return spark.read.format("delta").load(path)

def write_delta_overwrite(df, path:str):
    (df
     .write
     .format("delta")
     .mode("overwrite")
     .option("overwrite", "true")
     .save(path)
     )

#### Load Silver tables

In [0]:
hrrp_silver = read_delta(f"{SILVER}/hrrp_clean")
hosp_silver = read_delta(f"{SILVER}/hospital_info_clean")
mspb_silver = read_delta(f"{SILVER}/mspb_clean")
unplanned_visits_silver = read_delta(f"{SILVER}/unplanned_visits_clean")
ruca_silver = read_delta(f"{SILVER}/ruca_clean")


In [0]:
# Row counts for each table
print("HRRP rows:", hrrp_silver.count())
print("Hospital rows:", hosp_silver.count())
print("MSPB rows:", mspb_silver.count())
print("Unplanned rows:", unplanned_visits_silver.count())
print("RUCA rows:", ruca_silver.count())

#### Aggregate unplanned visits to facility level

Because one hospital can have multiple types of unplanned visits, we aggregate to the facility_id level to avoid "Row Explosion" during the join.

In [0]:
# Checking on the number of rows in each table before joining hospitals with HRRP
print("Hospital rows:", hosp_silver.count())
print("HRRP rows:", hrrp_silver.count())

In [0]:
unplanned_facility_agg = (
    unplanned_visits_silver
    .groupBy("facility_id")
    .agg(
        F.avg("score").alias("avg_unplanned_score"),
        F.sum("denominator").alias("total_unplanned_denominator"),
        F.sum("number_of_patients").alias("total_unplanned_patients"),
        F.sum("number_of_patients_returned").alias("total_unplanned_patients_returned")
    )
    .withColumn(
        "unplanned_return_rate",
        F.when(
            (F.col("total_unplanned_patients").isNotNull()) &
            (F.col("total_unplanned_patients") > 0),
            F.col("total_unplanned_patients_returned") / F.col("total_unplanned_patients")
        ).otherwise(F.lit(None))
    )
)

#### Prepare MSPB features

Rename score.

In [0]:
mspb_features = (
  mspb_silver
  .select("facility_id", F.col("score").alias("mspb_score"))
)

mspb_features.show(10)

#### Hospital Geodemographic Join
* Join hospital info with RUCA on zip_code

In [0]:
# Checking on the number of rows in each table before joining hospitals with HRRP
print("hosp_silver_columns: ",hosp_silver.columns)
print("\nruca_silver_columns: ",ruca_silver.columns)

In [0]:
# Merge hospital metadata with RUCA classification
hospital_geo = (
  hosp_silver
  .join(
    ruca_silver.select("zip_code", "ruca_code", "secondary_ruca_code"),
    on = "zip_code",
    how = "left"
  )
)

In [0]:
# Verify the join columns
hospital_geo.columns

Ruca bucketing

In [0]:
# Categorize rural vs urban based on RUCA definitions
hospital_geo = hospital_geo.withColumn(
  "ruca_bucket",
  F.when(F.col("ruca_code").isNull(), F.lit(None))
  .when(F.col("ruca_code") <=3.0, F.lit("metro"))
  .when(F.col("ruca_code") <=6.0, F.lit("micro"))
  .otherwise(F.lit("rural_small_town"))
)

In [0]:
hospital_geo.count()

#### Create joined Gold hospital feature table
* Base = HRRP (contains target)

In [0]:
# Build the final gold table
gold_joined = (
    hrrp_silver.alias("h")
    .join(
        hospital_geo.alias("hg"),
        on="facility_id",
        how="left"
    )
    .join(
        mspb_features.alias("m"),
        on="facility_id",
        how="left"
    )
    .join(
        unplanned_facility_agg.alias("u"),
        on="facility_id",
        how="left"
    )
)

In [0]:
# Select the columns for the final table to avoid duplication
gold_hospital_features = gold_joined.select(
    # identifiers / time
    F.col("facility_id"),
    F.col("h.facility_name").alias("facility_name"),
    F.col("h.state").alias("state"),
    F.col("h.start_date").alias("start_date"),
    F.col("h.end_date").alias("end_date"),
    F.col("hg.zip_code").alias("zip_code"),

    # HRRP target + clinical metrics
    F.col("h.measure_name").alias("measure_name"),
    F.col("h.number_of_discharges").alias("number_of_discharges"),
    F.col("h.number_of_readmissions").alias("number_of_readmissions"),
    F.col("h.predicted_readmission_rate").alias("predicted_readmission_rate"),
    F.col("h.expected_readmission_rate").alias("expected_readmission_rate"),
    F.col("h.excess_readmission_ratio").alias("excess_readmission_ratio"),

    # hospital attributes
    F.col("hg.hospital_type").alias("hospital_type"),
    F.col("hg.hospital_ownership").alias("hospital_ownership"),
    F.col("hg.emergency_services").alias("emergency_services"),
    F.col("hg.hospital_overall_rating").alias("hospital_overall_rating"),
    F.col("hg.meets_criteria_for_birthing_friendly_designation").alias("birthing_friendly_designation"),

    # geography
    F.col("hg.ruca_code").alias("ruca_code"),
    F.col("hg.secondary_ruca_code").alias("secondary_ruca_code"),
    F.col("hg.ruca_bucket").alias("ruca_bucket"),

    # cost + utilization
    F.col("m.mspb_score").alias("mspb_score"),
    F.col("u.avg_unplanned_score").alias("avg_unplanned_score"),
    F.col("u.total_unplanned_denominator").alias("total_unplanned_denominator"),
    F.col("u.total_unplanned_patients").alias("total_unplanned_patients"),
    F.col("u.total_unplanned_patients_returned").alias("total_unplanned_patients_returned"),
    F.col("u.unplanned_return_rate").alias("unplanned_return_rate")
)

In [0]:
gold_hospital_features.columns

#### Feature engineering

In [0]:
# Add additional features

gold_hospital_features = (
    gold_hospital_features

    # binary indicators
    .withColumn(
        "is_emergency_services",
        F.when(F.col("emergency_services") == "Yes", F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "has_hospital_rating",
        F.when(F.col("hospital_overall_rating").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "high_quality_hospital",
        F.when(F.col("hospital_overall_rating") >= 4, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "high_patient_volume",
        F.when(F.col("total_unplanned_patients") > 500, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "is_rural",
        F.when(F.col("ruca_code") >= 7, F.lit(1)).otherwise(F.lit(0))
    )

    # derived ratios/gaps
    .withColumn(
        "readmission_gap",
        F.when(
            F.col("predicted_readmission_rate").isNotNull() &
            F.col("expected_readmission_rate").isNotNull(),
            F.col("predicted_readmission_rate") - F.col("expected_readmission_rate")
        ).otherwise(F.lit(None))
    )
    .withColumn(
        "observed_readmission_rate",
        F.when(
            (F.col("number_of_discharges").isNotNull()) &
            (F.col("number_of_discharges") > 0) &
            F.col("number_of_readmissions").isNotNull(),
            F.col("number_of_readmissions") / F.col("number_of_discharges")
        ).otherwise(F.lit(None))
    )

    # classification label for later ML classification
    .withColumn(
        "high_readmission_flag",
        F.when(F.col("excess_readmission_ratio") > 1.0, F.lit(1)).otherwise(F.lit(0))
    )
)

Build a narrower ML-ready table 

In [0]:
gold_ml_features = gold_hospital_features.select(
    "facility_id",
    "facility_name",
    "state",
    "zip_code",
    "measure_name",
    "start_date",
    "end_date",

    "hospital_type",
    "hospital_ownership",
    "hospital_overall_rating",
    "birthing_friendly_designation",
    "is_emergency_services",
    "has_hospital_rating",
    "high_quality_hospital",
    "high_patient_volume",
    "is_rural",

    "mspb_score",
    "avg_unplanned_score",
    "total_unplanned_denominator",
    "total_unplanned_patients",
    "total_unplanned_patients_returned",
    "unplanned_return_rate",

    "ruca_code",
    "secondary_ruca_code",
    "ruca_bucket",

    "predicted_readmission_rate",
    "expected_readmission_rate",
    "readmission_gap",
    "observed_readmission_rate",

    # targets
    "excess_readmission_ratio",
    "high_readmission_flag"
)


Validation checks

In [0]:
print("Gold hospital features rows:", gold_hospital_features.count())
print("Gold ML features rows:", gold_ml_features.count())

gold_hospital_features.select(
    F.count("*").alias("rows"),
    F.sum(F.col("emergency_services").isNull().cast("int")).alias("null_emergency_services"),
    F.sum(F.col("hospital_overall_rating").isNull().cast("int")).alias("null_hospital_rating"),
    F.sum(F.col("total_unplanned_patients").isNull().cast("int")).alias("null_total_unplanned_patients"),
    F.sum(F.col("ruca_code").isNull().cast("int")).alias("null_ruca_code"),
    F.sum(F.col("mspb_score").isNull().cast("int")).alias("null_mspb_score")
).show()

# feature distributions
gold_hospital_features.groupBy("emergency_services").agg(F.count("*").alias("cnt")).orderBy("emergency_services").show()
gold_hospital_features.groupBy("hospital_overall_rating").agg(F.count("*").alias("cnt")).orderBy("hospital_overall_rating").show()
gold_hospital_features.groupBy("ruca_code").agg(F.count("*").alias("cnt")).orderBy("ruca_code").show()

In [0]:
# write to gold delta table
write_delta_overwrite(gold_hospital_features, f"{GOLD}/hospital_features")
write_delta_overwrite(gold_ml_features, f"{GOLD}/ml_features")